### 1.3.7.9. Numerical, Symbolic, and Automatic Differentiation

$$
\underbrace{\frac{f(x+h) - f(x-h)}{2h}}_{\text{numerical (finite difference)}}
\;\approx\;
\underbrace{f'(x)}_{\text{symbolic / automatic (exact)}} .
$$

**Explanation:**

There are three ways to obtain derivatives, with different trade-offs. *Symbolic* differentiation manipulates expressions to return an exact formula but can blow up in size. *Numerical* (finite-difference) differentiation only evaluates $f$ but suffers both truncation error ($O(h^2)$ for the central difference) and floating-point cancellation as $h \to 0$, so it is never exact. *Automatic* differentiation ([forward](./07_forward_mode_automatic_differentiation.ipynb) and [reverse](./08_reverse_mode_automatic_differentiation.ipynb)) evaluates derivatives to machine precision at a small multiple of the function cost. The standard practice is to use AD for gradients and a [finite-difference](../../../03_Optimization/05_Unconstrained_Algorithms/29_finite_difference_derivative_approximations.ipynb) check to guard against bugs (gradient checking).

**Properties:**
- Central differences have error $O(h^2)$; an overly small $h$ amplifies round-off, so an intermediate $h$ is best.
- Symbolic and AD agree to machine precision; the finite difference agrees only approximately.

**Numerical Example:**

Differentiate $f(x) = x^2 + \sin x$ at $x = 2$; the exact value is $f'(2) = 4 + \cos 2 \approx 3.583853$.

Central difference with $h = 10^{-5}$:
$$
\frac{f(2 + h) - f(2 - h)}{2h} \approx 3.583853 ,
$$

agreeing to about $10^{-10}$. Automatic differentiation returns $4 + \cos 2$ to full machine precision, and symbolic differentiation returns the formula $2x + \cos x$ exactly. The finite difference matches only up to its truncation/round-off error, illustrating why AD is preferred and the finite difference is kept for checking.

In [1]:
import sympy as sp
import math

x = sp.symbols("x")
f = x**2 + sp.sin(x)
point = 2

symbolic_derivative = sp.diff(f, x)
symbolic_value = float(symbolic_derivative.subs(x, point))

def f_numeric(value):
    return value**2 + math.sin(value)

step = 1e-5
central_difference = (f_numeric(point + step) - f_numeric(point - step)) / (2 * step)

print("symbolic f'(x)      =", symbolic_derivative)
print("symbolic f'(2)      =", symbolic_value)
print("central difference  =", central_difference)
print("difference          =", abs(symbolic_value - central_difference))

symbolic f'(x)      = 2*x + cos(x)
symbolic f'(2)      = 3.5838531634528574
central difference  = 3.583853163480199
difference          = 2.7341684472048655e-11


**Equivalent CasADi implementation:**

CasADi's automatic differentiation matches the symbolic value to machine precision; comparing it against the finite difference is exactly the gradient check used in practice.

In [2]:
import casadi as ca

x = ca.SX.sym("x")
f = x**2 + ca.sin(x)
automatic_derivative = ca.Function("df", [x], [ca.gradient(f, x)])

automatic_value = float(automatic_derivative(2.0))
step = 1e-5
f_function = ca.Function("f", [x], [f])
central_difference = float((f_function(2.0 + step) - f_function(2.0 - step)) / (2 * step))

print("automatic f'(2)    =", automatic_value)
print("central difference =", central_difference)
print("gradient-check gap =", abs(automatic_value - central_difference))

automatic f'(2)    = 3.5838531634528574
central difference = 3.583853163480199
gradient-check gap = 2.7341684472048655e-11


**References:**

[📘 Griewank, A., & Walther, A. (2008). *Evaluating Derivatives* (2nd ed.). SIAM.](https://epubs.siam.org/doi/book/10.1137/1.9780898717761)  
[📗 Baydin, A. G., Pearlmutter, B. A., Radul, A. A., & Siskind, J. M. (2018). *Automatic Differentiation in Machine Learning: a Survey*. JMLR 18(153).](https://jmlr.org/papers/v18/17-468.html)

---

[⬅️ Previous: Reverse-Mode Automatic Differentiation](./08_reverse_mode_automatic_differentiation.ipynb) | [Next: Optimization Problem ➡️](../../../03_Optimization/01_Optimization_Fundamentals/01_optimization_problem.ipynb)